# koilog — журнал силовых тренировок

`koilog` разбирает две выгрузки в одну базу SQLite, считает производные
метрики (1RM, рекорд, скользящее среднее, КОИ) по явным правилам и строит
программу роста нагрузки от последней реальной тренировки — только по тем
весам, которые есть в инвентаре зала.

Вся логика живёт в пакете `koilog` и вызывается из командной строки; этот
ноутбук — обзор поверх той же базы. Он импортирует **те же** функции, что и
CLI, поэтому расхождений между выводом здесь и в терминале нет.

```
python -m koilog load                       # разобрать CSV в sql/training.db
python -m koilog refresh                    # журнал + свежие данные часов в базу
python -m koilog stats                      # сводка по упражнениям за 3 месяца
python -m koilog stats --all                # вся история
python -m koilog plan "Жим гантелей"        # программа + периоды
python -m koilog weeks --exercise "Жим гантелей"   # недельные суммы КОИ
python -m koilog history "Жим гантелей" --from 2026-01-01 --to 2026-10-08
python -m koilog explain "Жим гантелей"     # разбор парсинга и расчёта
python -m koilog check                      # что в данных требует решения
```

Пакет держится на стандартной библиотеке Python 3.13. `matplotlib` и
`ipywidgets` нужны только этому ноутбуку и на CLI не влияют.

## Как читать числа

Короткий словарь, чтобы колонки не приходилось угадывать.

**1RM** — оценка максимального веса на одно повторение по формуле Brzycki
`вес × 36 / (37 − повторы)`. Выше 20 повторов не считается: экстраполяция
перестаёт иметь смысл.

**Для упражнений «тело + жилет»** (`Отжимания`) рабочий вес — это **суммарный**
вес: собственное тело плюс добавка. В таблицах ниже, где нужен вес, он уже
суммарный; в `explain` видно, как из «сырых» 20 кг жилета получается 90 кг
(70 тело + 20 жилет).

**Режим записи веса** в таблицах помечен в квадратных скобках: `[тело+груз]`
— записан суммарный вес (тело плюс добавка), `[только груз]` — одна добавка.
Это подпись, а не новая метрика: 1RM, КОИ и объём считаются как раньше.

| колонка | что значит |
|---|---|
| подх | подходов упражнения в выбранном окне |
| лучший | лучший 1RM **внутри окна** |
| рекорд | лучший 1RM за **всю историю** — база для следующей колонки |
| посл. | 1RM последней тренировки в окне |
| Δ к рекорду | `посл. − рекорд`: 0 = на рекорде, минус = отставание |
| МА30 | скользящее среднее 1RM за **30 дней по этому упражнению** (база КОИ) |
| объём | сумма `вес × повторы` по подходам, в килограммо-повторах |

Раньше на месте «Δ к рекорду» стояла дельта к предыдущей тренировке. Она
честно равнялась нулю на двух одинаковых тренировках подряд (и это выглядело
как поломка), поэтому теперь дельта считается к рекорду, а изменение к прошлой
тренировке печатает `koilog plan` — с датой той тренировки.

**КОИ** — «коэффициент относительной интенсивности», мера усталостной нагрузки.
Формула оригинальная: `КОИ = Σ повторы / (100 − интенсивность%)`, где
интенсивность = `вес / база 1RM`. Делить на 100 обязательно: интенсивность —
доля, а формула выведена в процентах. Выше 98 % от базы КОИ не считается:
при интенсивности около 100 % знаменатель уходит в ноль.

**КОИ считается по каждому упражнению отдельно**, а не по группе: у жима и
тяги в одной группе разные базы 1RM, и общая сумма теряла бы масштаб. Поэтому
и недельные суммы, и суммы за тренировку — по упражнениям.

## Обновление базы


In [ ]:
# Шаг обновления: журнал (sets) + свежие данные часов и производные
# в базу (журнал -> выгрузка -> нагрузка -> эпизоды), ML-блок не трогаем.
# Журнал грузится из data/journal.csv по слепку mtime:size, поэтому свежая
# тренировка появляется сама; отдельный `python -m koilog load` не нужен.
# Печатаем только имя файла и даты: абсолютный путь с именем пользователя
# не должен попадать в сохранённый вывод ноутбука.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

from koilog import etl, muscle_load, stats
from koilog.health import refresh, text

conn = etl.connect(etl.DEFAULT_DB)
print(etl.DEFAULT_DB.name)
print(refresh.format_freshness(refresh.run(conn)))


In [ ]:
# Что вообще загрузилось: сколько подходов дал каждый источник и за какой срок.
# То же печатает `python -m koilog load` (блок «источник ... подходов»).
for name, block in etl.summary(conn)["by_source"].items():
    print(f"{name:14} {block['sets']:6} подходов  {block['days']:4} дней  "
          f"{block['first_day']} ... {block['last_day']}")

In [ ]:
# Сводка упражнений за окно - то же, что печатает `python -m koilog stats`.
# Окно здесь - три месяца (как у `stats` без ключа --all): упражнения, которыми
# давно не занимались, в неё не попадают.
# Подписи колонок: подх - подходов, лучший/рекорд/посл. - 1RM, Δ к рекорду -
# отставание последней тренировки от рекорда, МА30 - среднее за 30 дней
# по упражнению, объём - вес x повторы.
# УЧИТЫВАТЬ_РАЗМИНКУ включает псевдо-упражнение «разминка» в счётчики
# (в таблице его не видно: она фильтрует по 1RM, а у разминки его нет).
# Группа подписана общим резолвером витрин: ДЕЛЬТЫ режутся на передние/
# средние/задние (NU-138 п.4). Числа от подписи не зависят.
since = stats.since_for_months(conn, 3)
УЧИТЫВАТЬ_РАЗМИНКУ = False
print(stats.format_report(
    stats.exercise_report(
        conn, order="best_rm", limit=15, with_rm_only=True, since=since,
        include_warmup=УЧИТЫВАТЬ_РАЗМИНКУ),
    group_label=muscle_load.group_label))
total, working = stats.window_sets(conn, since)
if УЧИТЫВАТЬ_РАЗМИНКУ:
    print(f"подходов в окне: {total} (без разминки: {working})")
else:
    print(f"подходов в окне: {working} (с разминкой: {total})")
print("лучший 1RM окна, рекорд за всю историю, последний 1RM и отставание "
      "от рекорда; МА30 - база КОИ по упражнению.")


### Что было за последний месяц

Список имён, которые реально использовались в последние 4 недели — перед ручным вводом. Полное имя видно целиком: защита от опечаток и дублей на вводе.

In [ ]:
# Список имён за последние 28 дней: что реально вводилось.
print(stats.format_recent_exercises(stats.recent_exercises(conn)))


In [ ]:
# Одно поле на весь ноутбук: его читают «Программа роста», «Переходы»,
# «История» и «График 1RM». Правьте здесь, а не в каждой ячейке отдельно.
# Литерал явный, а не «первая строка списка»: в списке выше есть кардио и
# «Разминка», у которых нет 1RM, - брать оттуда значение по умолчанию нельзя.
УПРАЖНЕНИЕ = "Жим гантелей лёжа"
print(f"упражнение для блоков ниже: «{УПРАЖНЕНИЕ}»")


In [ ]:
# Программа роста нагрузки от последней реальной тренировки: стартовые вес и
# повторы берутся из базы, список весов - из инвентаря зала.
# В шапке видно рекорд, изменение к прошлой тренировке (с её датой) и МА30 -
# базу КОИ. Если в инвентаре нет записанного веса, программа стартует
# с ближайшего не тяжелее и пишет об этом строкой «~».
# Переходы на следующий вес программа больше не печатает - они отдельной
# ячейкой ниже.
# Упражнение - из общего поля «УПРАЖНЕНИЕ» выше; здесь своего имени нет.
print(stats.format_plan(stats.progression_from_db(conn, УПРАЖНЕНИЕ)))


In [ ]:
# Переходы на следующий вес: где заканчивается «волна» на текущем снаряде и на
# какой вес программа переходит дальше. Колонка «дельта» - насколько 1RM нового
# веса отличается от текущего (обычно вниз: новый вес берётся с 6 повторов).
# Это отдельная ячейка, а не часть «программы роста»: там те же переходы
# выводились дважды.
plan = stats.progression_from_db(conn, УПРАЖНЕНИЕ)
print(stats.format_transitions(plan.wave))


In [ ]:
# Недельные суммы КОИ - по КАЖДОМУ упражнению отдельно (оригинальная логика
# КОИ). Колонка «КОИ» - сумма подходов недели по упражнению, «режим» - полоса
# по этой сумме. Отдельной колонки «на подход» нет: полоса задана именно для
# суммы, а усреднение её размывало.
# Полосы недельного КОИ: < 2 лёгкая, 2…3 тяжёлая, 3…4 очень тяжёлая,
# больше 4 «Вы сошли с ума??».
print(stats.format_weekly_koi(stats.weekly_koi(conn, limit=2)))

In [ ]:
# Недельные суммы КОИ по группам - простое среднее по упражнениям группы за
# неделю (не взвешенное объёмом). Группа подписана тем же резолвером, что и
# Сводка: ДЕЛЬТЫ видны как передние/средние/задние.
# Числа - производные от поупражненных КОИ выше: веса участия не вводятся.
print(stats.format_weekly_koi_by_group(
    stats.weekly_koi_by_group(conn, limit=2,
                              group_label=muscle_load.group_label)))


## Нагрузка по мышцам

Кольцо нагрузки (подходы, поделённые `1/N` на мышцы упражнения) и свежесть:
когда каждая мышца последний раз получала стимул. Это витрина 4.14,
display-only — в решения (гейт, рампа, дайджест) она не входит, поэтому и
живёт рядом с КОИ, а не в health-ноутбуке.


In [ ]:
# Нагрузка по мышцам (4.14, display-only): кольцо по подходам 1/N и
# тоннаж только внутри режима. Пререг: docs/4-14-prereg.md.
import datetime

%matplotlib inline
import matplotlib.pyplot as plt

from koilog import muscle_load

muscle_anchor = muscle_load.freshness(conn).anchor
muscle_since = (datetime.date.fromisoformat(muscle_anchor)
                - datetime.timedelta(days=27)).isoformat()
week = muscle_load.week_key(muscle_anchor)
ring_month = muscle_load.ring(conn, since=muscle_since,
                              until=muscle_anchor)
ring_week = muscle_load.week_ring(conn, week)
iron = muscle_load.tonnage_by_muscle(conn, muscle_load.IRON,
                                     since=muscle_since,
                                     until=muscle_anchor)
body = muscle_load.tonnage_by_muscle(conn, muscle_load.BODYWEIGHT,
                                     since=muscle_since,
                                     until=muscle_anchor)
print(f"нагрузка по мышцам: неделя {week} и 28 дней до {muscle_anchor}")
print(text.table(
    ["мышца", "неделя, %", "28 дн, %", "железо, кг", "своё тело, кг"],
    [[load.muscle,
      f"{ring_week.of(load.muscle).share * 100:.1f}",
      f"{load.share * 100:.1f}",
      f"{iron.of(load.muscle).kg:.0f}",
      f"{body.of(load.muscle).kg:.0f}"] for load in ring_month.loads]))

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh([load.muscle for load in ring_month.loads],
        [load.share * 100 for load in ring_month.loads], color="#4c72b0")
ax.set_xlabel("доля подходов 1/N, % (28 дней)")
ax.set_title(f"нагрузка по мышцам, 28 дней до {muscle_anchor}")
fig.tight_layout()
plt.show()


In [ ]:
# Свежесть мышц (4.14): порог 48 ч (дней <= 1), якорь = последняя дата
# данных; сырое «дней» всегда рядом, хвосты не прячутся.
fresh = muscle_load.freshness(conn)

rows = []
for row in fresh.muscles:
    if row.days is None:
        status = "нет стимула"
    elif row.recovered:
        status = "восстановлена"
    else:
        status = "восстанавливается"
    rows.append([row.muscle, row.last or "—",
                 "—" if row.days is None else row.days, status, row.sets])
print(f"свежесть на {fresh.anchor}")
print(text.table(["мышца", "последний стимул", "дней", "статус",
                  "подходов"], rows))

fig, ax = plt.subplots(figsize=(7, 5))
for i, row in enumerate(fresh.muscles):
    days = row.days if row.days is not None else 0
    if row.days is None:
        color = "0.7"
    elif row.recovered:
        color = "#55a868"
    else:
        color = "#dd8452"
    ax.barh(i, min(days, 14), color=color)
    ax.text(min(days, 14) + 0.1, i, f"{days}", va="center", fontsize=8)
ax.set_yticks(range(len(fresh.muscles)))
ax.set_yticklabels([row.muscle for row in fresh.muscles])
ax.invert_yaxis()
ax.set_xlim(0, 15)
ax.set_xlabel("дней с последнего стимула (полоса до 14)")
ax.set_title(f"свежесть мышц на {fresh.anchor}")
fig.tight_layout()
plt.show()


### История упражнения за интервал

Интервал задаётся двумя строками ISO-дат `ГГГГ-ММ-ДД`, **обе границы
включительно**. `None` в любой из них означает «без границы с этой стороны»:
`ИНТЕРВАЛ_ОТ = None` и `ИНТЕРВАЛ_ДО = None` берут всю историю упражнения.

Направление вывода задаёт `ОБРАТНЫЙ_ПОРЯДОК`: `False` — от первой даты к
последней, `True` — от последней к первой (и тогда `limit` оставляет самые
свежие тренировки). Тот же отчёт в терминале:
`python -m koilog history "УПРАЖНЕНИЕ" --from ДАТА --to ДАТА` (флаг `--reverse`
разворачивает порядок).

In [ ]:
import datetime

# Меняются три строки ниже. Даты - включительно, в ISO (ГГГГ-ММ-ДД).
# None = границы нет: от None - с первой записи, до None - по последнюю.
# ОБРАТНЫЙ_ПОРЯДОК=True покажет те же тренировки от последней даты к первой.
# Упражнение - из общего поля «УПРАЖНЕНИЕ» выше; своего имени здесь нет.
# е/ё: если точного имени в базе нет, stats.resolve_exercise подберёт
# хранимый вариант (ё→е/е→ё). Шапка печатает разрешённое имя.
ИНТЕРВАЛ_ОТ = datetime.date(2026, 1, 1)   # пример: с 1 января 2026
ИНТЕРВАЛ_ДО = None                        # None - по последнюю тренировку
ОБРАТНЫЙ_ПОРЯДОК = False                  # False - от старых; True - от новых


def _iso(value):
    # date -> ISO-строка, None -> None.
    return None if value is None else value.isoformat()


RESOLVED = stats.resolve_exercise(conn, УПРАЖНЕНИЕ)
rows = stats.history(
    conn, RESOLVED,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
)
print(stats.format_history(
    rows, RESOLVED,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
))
# «верхний подход» - лучший подход тренировки; у записей «тело+груз» он
# показан суммой с компонентами (тело+добавка), у «только груз» - одной
# добавкой. 1RM - его 1RM; Δ 1RM - к прошлой тренировке упражнения;
# КОИ - сумма подходов тренировки; режим - по KOI_BANDS.


In [ ]:
# Разбор одного упражнения: сначала - как читаются выгрузки (кодировка,
# разделитель, приоритет дат, правила), затем - от «сырого» значения в файле
# до КОИ, подход за подходом.
# Это ответ на вопрос «откуда взялось 111.7 у Отжиманий»: в колонках видно
# «сырой 20 -> режим vest_only -> свой вес 70 + добавка 20 -> вес 90 -> 1RM».
# Таблицу формирует общая функция stats.format_explain - тот же вид, что в
# `python -m koilog explain "Отжимания"`.
print("=== как читаются выгрузки ===")
for note in etl.parse_notes():
    print(f"  - {note}")

audit = etl.exercise_audit(conn, УПРАЖНЕНИЕ)
if audit is None:
    print(f"\nупражнение «{УПРАЖНЕНИЕ}» в базе не найдено")
else:
    print(f"\n=== разбор «{audit.exercise}» ===")
    print(f"подходов всего: {audit.total}, с 1RM: {audit.with_rm}; "
          f"на времени: {audit.timed}, пропущено: {audit.skipped}, "
          f"выбросов: {audit.outlier}; период {audit.first_day} … {audit.last_day}")
    print("режимы веса: " + ", ".join(f"{n} ({c})" for n, c in audit.weight_modes))

    print("\n=== последние подходы: от сырого значения до КОИ ===")
    print(stats.format_explain(stats.explain_sets(conn, УПРАЖНЕНИЕ, limit=8)),
          end="")

## График 1RM

`matplotlib` и `ipywidgets` — единственные нестандартные библиотеки проекта, и
нужны они только ноутбуку: в пакете `koilog` графики остаются ASCII
(`koilog weeks`, `koilog plan`).

```
pip install matplotlib ipywidgets
```

Ячейка ниже рисует **ровно один** график 1RM для выбранного упражнения за
выбранный период. Если `ipywidgets` установлен — выбор упражнения и дат
делается полями; если нет — тем же кодом (`УПРАЖНЕНИЕ`, `СТАРТ`, `ФИНИШ`).

Раньше здесь был баг: каждая смена поля рисовала новый график **поверх**
предыдущего, и в сохранённом выводе их накапливалось по три-четыре. Причина —
`draw_rm` не закрывал фигуру, и pyplot держал все открытые кадры. Теперь
фигура закрывается в `finally`, поэтому интерактивный и статичный варианты
ведут себя одинаково: на экране всегда одна кривая.


In [ ]:
# Абсолютный 1RM выбранного упражнения за выбранный период.
#
# Как задать диапазон:
#   * поле «упражнение» - выпадающий список всех упражнений с посчитанным 1RM;
#   * поле «с» и «по» - календарные даты начала и конца, обе включительно.
# Упражнение берётся из общего поля «УПРАЖНЕНИЕ» в начале ноутбука: своего
# литерала здесь нет, поэтому одна правка меняет и таблицы, и график.
# Если введённого имени нет среди упражнений с 1RM, ячейка честно об этом
# скажет и НЕ подменит его первым из списка.
#
# Главное правило ячейки: график всегда ОДИН. draw_rm закрывает фигуру через
# plt.close(fig) в finally - без этого pyplot копит открытые кадры, и каждая
# смена поля (или повторный запуск ячейки) добавляла бы ещё одну кривую:
# автор доходил до четырёх графиков в одном выводе.
#
# ipywidgets необязателен: без него ячейка просто рисует один статичный кадр по
# переменным выше - ноутбук не ломается.
import datetime

%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Упражнения, у которых вообще есть 1RM (для выбора в поле).
NAMES = [row.exercise
         for row in stats.exercise_report(conn, with_rm_only=True,
                                          order="exercise")]

СТАРТ = datetime.date(2015, 1, 1)
ФИНИШ = datetime.date.today()


def draw_rm(exercise, start, end):
    # Нарисовать ровно один график 1RM за интервал [start, end] включительно.
    # Виджет вызывает её при каждом изменении поля, поэтому здесь обязательно
    # закрыть фигуру - иначе кривые складываются друг на друга.
    since = start.isoformat() if start else None
    until = end.isoformat() if end else None
    rows = stats.history(conn, exercise, since=since, until=until)

    fig, ax = plt.subplots(figsize=(11, 4.6))
    try:
        if not rows:
            ax.set_title(f"нет 1RM: «{exercise}» за {since or 'начало'} … "
                         f"{until or 'конец'}")
        else:
            dates = [r.date for r in rows]
            values = [r.rm_session if r.rm_session is not None else float("nan")
                      for r in rows]
            ax.plot(dates, values, marker="o", ms=4, lw=1.6, color="#4c72b0")
            ax.set_title(f"1RM: {exercise} ({dates[0]} … {dates[-1]})")
            ax.set_xlabel("дата")
            ax.set_ylabel("1RM, кг")
            last = values[-1]
            if last == last:  # не NaN
                ax.annotate(f"{last:.1f} кг", xy=(dates[-1], last),
                            xytext=(-8, 10), textcoords="offset points",
                            ha="right", fontweight="bold")
            fig.autofmt_xdate(rotation=30)
        plt.show()
    finally:
        plt.close(fig)


try:
    import ipywidgets as wd
    from IPython.display import display
except ImportError:
    wd = None

if УПРАЖНЕНИЕ not in NAMES:
    print(f"«{УПРАЖНЕНИЕ}» нет среди упражнений с 1RM (их {len(NAMES)}): "
          f"впишите имя из списка выше или выберите его в поле - подменять "
          f"другим упражнением ячейка не будет.")
elif wd is None:
    # Виджетов нет: один статичный кадр по переменным выше.
    draw_rm(УПРАЖНЕНИЕ, СТАРТ, ФИНИШ)
else:
    f_ex = wd.Dropdown(options=NAMES, value=УПРАЖНЕНИЕ,
                       description="упражнение")
    f_from = wd.DatePicker(value=СТАРТ, description="с")
    f_to = wd.DatePicker(value=ФИНИШ, description="по")
    ui = wd.VBox([
        wd.HBox([f_ex]),
        wd.HBox([f_from, f_to]),
    ])
    out = wd.interactive_output(
        draw_rm,
        {"exercise": f_ex, "start": f_from, "end": f_to},
    )
    display(ui, out)


## О формулах и о том, что осталось открытым

**Что уже решено** — это не открытые вопросы, а действующие правила:

- **КОИ по упражнению.** Формула `Σ повторы / (100 − интенсивность%)`
  применяется к каждому упражнению отдельно — и за неделю, и за тренировку.
  Пороги полос вынесены в `koilog.core`: за неделю `< 2 / 2…3 / 3…4 / > 4`,
  за одну тренировку `< 0.4 / 0.4…1 / 1…2 / > 2`. Раньше недельные суммы
  складывались по группам, из-за чего жим и тяга делили одну сумму; теперь это
  исправлено.
- **Масштаб КОИ** честный (доли, а не проценты: тренировка в 1,1–1,3 — это
  `Load`, а не `Crazy`). Недельные суммы сравниваются по полосам
  `core.KOI_BANDS_WEEK`, которые заданы именно для суммы.
- **Вес собственного тела** больше не константа 70: он оценивается по годам из
  упражнений с суммарным весом (подтягивания, отжимания на брусьях) — сначала
  «чистый пол», иначе мода, затем расхождение между упражнениями года
  разрешается вниз, потому что жилет добавляет вес. Переопределить:
  `python -m koilog load --bodyweight-year 2021=82`.

**Что остаётся открытым:**

- **47 подходов 2018–2021** записаны весом жилета вместо суммарного
  (`Подтягивания` 25, 31,7, 37,3 кг при теле 67,4). Правила для «слишком мало»
  нет — выдумывать порог значит выкидывать настоящие подходы, поэтому строки
  помечены в `koilog check` как неправдоподобная добавка, но остаются в базе.
- **26 выбросов** (`is_outlier = 1`: `pushups_max` 20, `bodyweight_max` 6)
  исключены из метрик и витрин. Это не вопрос, а правило; число стоит держать
  перед глазами — его печатает `koilog check` ниже.


## Приложение: проверка данных

Служебный аудит качества данных: нераспознанные имена, подозрительные секунды, аномалии веса, выбросы.

In [ ]:
# Проблемные данные: что требует ручного решения (нераспознанные упражнения,
# подозрительное время, неправдоподобная добавка, вес тела по годам, выбросы).
# То же печатает `python -m koilog check`.
print(etl.quality_report(conn).format(limit=10), end="")